# llm-assist showcase (end-to-end runbook)

This notebook is the single execution flow for setup, data analysis, training demos, API checks, testing, live evaluation, and report asset generation.

## 0) Setup

Initialize environment helpers and resolve the repository root.

In [ ]:
import sys, os
print(sys.executable)
print(os.getcwd())

In [ ]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

import requests

# Works whether notebook starts in repo root or notebooks/
cwd = Path.cwd().resolve()
ROOT = cwd if (cwd / "app").is_dir() else cwd.parent
assert (ROOT / "app").is_dir(), f"Could not find repo root from {cwd}"
os.chdir(ROOT)  # make all relative paths consistent
print("Repo root:", ROOT)
print("Python:", sys.version)
print("Executable:", sys.executable)


## 1) Project context

Show core project summary and required source files used for assignment alignment.

In [ ]:
print((ROOT / "README.md").read_text(encoding="utf-8").splitlines()[0])
print()
print("Key sources:")
print("-", ROOT / "literature_review.md")
print("-", ROOT / "data" / "golden" / "README.md")


In [ ]:
lit_path = ROOT / "literature_review.md"
if not lit_path.is_file():
    print("Missing:", lit_path)
else:
    text = lit_path.read_text(encoding="utf-8")
    print(text[:6000])
    if len(text) > 6000:
        print("\n... [truncated; open full file at]", lit_path)


In [ ]:
golden_readme = ROOT / "data" / "golden" / "README.md"
if not golden_readme.is_file():
    print("Missing:", golden_readme)
else:
    body = golden_readme.read_text(encoding="utf-8")
    print("=" * 72)
    print("Golden dataset README:", golden_readme)
    print(body[:4500])
    if len(body) > 4500:
        print("\n... [truncated]")

print("\nTask coverage used in this notebook:")
print("- Triage: category/priority labels from golden set")
print("- Summarization: golden_summary for ROUGE-L")
print("- Quality: rubric mean score tracking")
print("- Sentiment: emitted by triage model during runtime")


## 2) Exploratory Data Analysis (EDA)

Run `scripts/run_eda.py` and visualize generated plots.

In [ ]:
eda_out = ROOT / "artifacts" / "eda"

# Run EDA script (requires: pip install -e ".[eda]")
subprocess.check_call(
    [sys.executable, "scripts/run_eda.py"],
    cwd=str(ROOT),
    env={**os.environ, "PYTHONUNBUFFERED": "1"},
)

print("EDA output dir:", eda_out)
print("Files:")
for p in sorted(eda_out.glob("*.png")):
    print("-", p.name)


In [ ]:
# Display figures inline
from IPython.display import Image, display

for p in sorted((ROOT / "artifacts" / "eda").glob("*.png")):
    display(Image(filename=str(p)))


## 3) Offline evaluation

Run evaluation script and print metrics summary from artifacts.

In [ ]:
subprocess.check_call(
    [
        sys.executable,
        "scripts/run_offline_eval.py",
        "--data",
        "data/golden/eval_set.jsonl",
    ],
    cwd=str(ROOT),
    env={
        **os.environ,
        # settings validation in CI expects a long enough secret
        "APP_SECRET_KEY": os.environ.get("APP_SECRET_KEY", "test-secret-key-minimum-16chars"),
        "QUALITY_POLICY_CONTEXT_TOP_K": os.environ.get("QUALITY_POLICY_CONTEXT_TOP_K", "0"),
        "TRIAGE_POLICY_CONTEXT_TOP_K": os.environ.get("TRIAGE_POLICY_CONTEXT_TOP_K", "0"),
        "PYTHONUNBUFFERED": "1",
    },
)

summary_path = ROOT / "artifacts" / "eval" / "summary.md"
print(summary_path.read_text(encoding="utf-8"))


## 4) Deep learning demos

Train transformer-based classifier and baseline classifier artifacts used in analysis.

In [ ]:
demo_csv = ROOT / "artifacts" / "demo_tickets.csv"

def write_demo_csv(path: Path) -> None:
    # 5 categories matching app.models.domain.Category
    rows = [
        ("charged twice for subscription", "billing"),
        ("refund not received for invoice", "billing"),
        ("cannot log in password reset fails", "authentication"),
        ("2FA code not working login blocked", "authentication"),
        ("export button crashes chrome", "technical_bug"),
        ("app freezes on settings screen", "technical_bug"),
        ("please add dark mode", "feature_request"),
        ("can you add SSO support", "feature_request"),
        ("what are your support hours", "general_inquiry"),
        ("where can I find API docs", "general_inquiry"),
    ]

    path.parent.mkdir(parents=True, exist_ok=True)
    lines = ["text,category\n"] + [f"{t},{c}\n" for t, c in rows]
    path.write_text("".join(lines), encoding="utf-8")

if not demo_csv.is_file():
    write_demo_csv(demo_csv)

print("Training CSV:", demo_csv)
print(demo_csv.read_text(encoding="utf-8"))


In [ ]:
model_dir = ROOT / "artifacts" / "triage_roberta_demo"

# Keep this short for a classroom demo
cmd = [
    sys.executable,
    "scripts/train_triage_transformer.py",
    "--data",
    str(demo_csv),
    "--out",
    str(model_dir),
    "--model",
    "roberta-base",
    "--epochs",
    "1",
    "--batch-size",
    "4",
]

print("Running:", " ".join(cmd))
subprocess.check_call(cmd, cwd=str(ROOT), env={**os.environ, "PYTHONUNBUFFERED": "1"})

metrics_path = model_dir / "train_metrics.json"
print("Checkpoint dir:", model_dir)
print("Metrics path:", metrics_path)
print(metrics_path.read_text(encoding="utf-8"))


## 5) API end-to-end pipeline checks

Start uvicorn in-notebook, call endpoints, and shut down cleanly.

In [ ]:
API_BASE = "http://127.0.0.1:8000"

# Start uvicorn as a subprocess
env = {
    **os.environ,
    "APP_SECRET_KEY": os.environ.get("APP_SECRET_KEY", "test-secret-key-minimum-16chars"),
    "QUALITY_POLICY_CONTEXT_TOP_K": os.environ.get("QUALITY_POLICY_CONTEXT_TOP_K", "0"),
    "TRIAGE_POLICY_CONTEXT_TOP_K": os.environ.get("TRIAGE_POLICY_CONTEXT_TOP_K", "0"),
    # Enable the optional encoder hint we just trained (safe even if deps missing; service will skip)
    "TRIAGE_TRANSFORMER_ENABLED": "true",
    "TRIAGE_TRANSFORMER_MODEL_DIR": str(model_dir),
    "PYTHONUNBUFFERED": "1",
}

uvicorn_cmd = [sys.executable, "-m", "uvicorn", "app.main:app", "--host", "127.0.0.1", "--port", "8000"]
print("Starting:", " ".join(uvicorn_cmd))
proc = subprocess.Popen(
    uvicorn_cmd,
    cwd=str(ROOT),
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

# Wait for /health
deadline = time.time() + 30
last_err = None
while time.time() < deadline:
    try:
        r = requests.get(f"{API_BASE}/api/v1/health", timeout=1.5)
        if r.status_code == 200:
            print("API ready:", r.json())
            break
    except Exception as e:
        last_err = e
    time.sleep(0.5)
else:
    raise RuntimeError(f"API did not become ready. Last error: {last_err}")


In [ ]:
import os
os.environ["LLM_PROFILE"] = "ollama"
os.environ["LLM_PROVIDER"] = "ollama"   # defensive
os.environ.pop("ANTHROPIC_API_KEY", None)
os.environ["LLM_TIMEOUT_SECONDS"] = "120"   # avoid 30s timeout on local model

In [ ]:
def post_json(path: str, payload: dict, *, timeout: int = 210, retries: int = 2) -> dict:
    last_response = None
    for attempt in range(retries + 1):
        r = requests.post(f"{API_BASE}{path}", json=payload, timeout=timeout)
        last_response = r
        if r.ok:
            return r.json()
        if attempt < retries:
            time.sleep(1.5)

    assert last_response is not None
    print(f"{path} failed ({last_response.status_code}):")
    try:
        print(json.dumps(last_response.json(), indent=2))
    except Exception:
        print(last_response.text)
    last_response.raise_for_status()
    return {}

sample_ticket = "I was charged twice this month and need a refund immediately."

# Warm-up call (reduces first-call latency variance on local/hosted models)
_ = post_json("/api/v1/triage", {"ticket_text": sample_ticket, "include_policy_context": False})

triage = post_json("/api/v1/triage", {"ticket_text": sample_ticket, "include_policy_context": False})
print("/triage:\n", json.dumps(triage, indent=2))

quality = post_json(
    "/api/v1/quality",
    {
        "ticket_text": sample_ticket,
        "agent_response": "Sorry about that. I opened a billing case and will refund within 24 hours.",
        "include_policy_context": False,
    },
)
print("\n/quality:\n", json.dumps(quality, indent=2))

pipeline = post_json(
    "/api/v1/pipeline",
    {
        "ticket_text": sample_ticket,
        "agent_response": "Sorry about that. I opened a billing case and will refund within 24 hours.",
    },
)
print("\n/pipeline:\n", json.dumps(pipeline, indent=2))

summ = post_json(
    "/api/v1/summarize",
    {
        "turns": [
            {"role": "customer", "content": "The app freezes when I open settings."},
            {
                "role": "agent",
                "content": "Please clear cache and update to 2.4. I will follow up tomorrow.",
            },
        ]
    },
)
print("\n/summarize:\n", json.dumps(summ, indent=2))

rag = post_json("/api/v1/rag/context", {"query": "refund timeline", "top_k": 2})
print("\n/rag/context:\n", json.dumps(rag, indent=2))


In [ ]:
# Shutdown uvicorn
try:
    proc.terminate()
    proc.wait(timeout=10)
except Exception:
    proc.kill()

# Print last ~200 lines of server output for debugging
if proc.stdout is not None:
    out = proc.stdout.read()
    lines = out.splitlines()[-200:]
    print("\n".join(lines))

print("Server stopped.")


## 6) Test suite execution

Run unit and integration checks from notebook.

In [ ]:
import subprocess, os
subprocess.run(
    [
        "pytest",
        "tests/unit/test_intent_fallback_service.py",
        "tests/unit/test_triage_service.py",
        "tests/integration/test_pipeline_e2e.py",
        "tests/integration/test_pipeline_label_recovery.py",
        "-q",
        "--no-cov",
    ],
    cwd=str(ROOT),   # or "/Users/aryamandev/.../llm-assist"
    check=False,
)

In [ ]:
def run_cmd(cmd: list[str], *, env_overrides: dict[str, str] | None = None, check: bool = True) -> subprocess.CompletedProcess:
    env = os.environ.copy()
    if env_overrides:
        env.update(env_overrides)
    print("Running:", " ".join(cmd))
    cp = subprocess.run(cmd, cwd=str(ROOT), env=env, text=True, capture_output=True)
    if cp.stdout.strip():
        print(cp.stdout)
    if cp.stderr.strip():
        print(cp.stderr)
    if check and cp.returncode != 0:
        raise RuntimeError(f"Command failed ({cp.returncode}): {' '.join(cmd)}")
    return cp

_ = run_cmd(
    [
        sys.executable,
        "-m",
        "pytest",
        "tests/unit/test_intent_fallback_service.py",
        "tests/unit/test_triage_service.py",
        "tests/integration/test_pipeline_e2e.py",
        "tests/integration/test_pipeline_label_recovery.py",
        "-q",
        "--no-cov",
    ]
)


## 7) Live-only evaluation analytics

Generate live metrics, confusion matrix, and minority-class views.

In [ ]:
import shutil
import pandas as pd

metrics_path = ROOT / "artifacts" / "eval" / "metrics.json"
summary_path = ROOT / "artifacts" / "eval" / "summary.md"

# Live eval (canonical for reporting)
_ = run_cmd(
    [sys.executable, "scripts/run_offline_eval.py", "--data", "data/golden/eval_set.jsonl"],
    env_overrides={"EVAL_LLM": "1"},
)
live_metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
live_copy = ROOT / "artifacts" / "eval" / "metrics_live.json"
live_copy.write_text(json.dumps(live_metrics, indent=2), encoding="utf-8")
if summary_path.exists():
    shutil.copy(summary_path, ROOT / "artifacts" / "eval" / "summary_live.md")

tri_cat = live_metrics.get("triage_category", {})
tri_pri = live_metrics.get("triage_priority", {})

rows = [
    ("mode", live_metrics.get("mode")),
    ("triage_category_accuracy", tri_cat.get("accuracy")),
    ("triage_category_micro_f1", tri_cat.get("micro_f1")),
    ("triage_category_macro_f1", tri_cat.get("macro_f1")),
    ("triage_priority_accuracy", tri_pri.get("accuracy")),
    ("triage_priority_micro_f1", tri_pri.get("micro_f1")),
    ("triage_priority_macro_f1", tri_pri.get("macro_f1")),
    ("quality_mean_score", live_metrics.get("quality", {}).get("mean_score")),
    ("summarize_mean_rouge_l_f1", live_metrics.get("summarize", {}).get("mean_rouge_l_f1")),
]
print("Live evaluation metrics:")
for metric, val in rows:
    print(f"- {metric}: {val}")

cm = pd.DataFrame(tri_cat.get("confusion_matrix", {})).T
if not cm.empty:
    cm = cm.fillna(0).astype(int)
    print("\nTriage category confusion matrix (true rows x predicted columns):")
    display(cm)

minority = tri_cat.get("minority_class_performance", {})
if minority:
    print("\nMinority-class performance (triage category):")
    display(pd.DataFrame(minority).T)

print("\nSaved:")
print("-", live_copy)
print("-", ROOT / "artifacts" / "eval" / "summary_live.md")


## 8) Presentation/report artifacts

Regenerate EDA figures, PPTX deck, and updated PDF report.

In [ ]:
from IPython.display import Image, display

# Regenerate EDA plots
_ = run_cmd([sys.executable, "scripts/run_eda.py"], check=False)

# Regenerate presentation + updated PDF report from current artifacts
_ = run_cmd([sys.executable, "scripts/generate_presentation_assets.py"])

out_ppt = ROOT / "LLM_Assist_Final_Presentation_50_Slides.pptx"
out_pdf = ROOT / "LlmCustomerSupport_project_report_updated.pdf"

print("Generated assets:")
print("-", out_ppt)
print("-", out_pdf)
print("-", ROOT / "artifacts" / "eval" / "metrics_live.json")
print("-", ROOT / "artifacts" / "eval" / "summary_live.md")

print("\nEDA figures:")
for p in sorted((ROOT / "artifacts" / "eda").glob("*.png")):
    print("-", p.name)
    display(Image(filename=str(p)))
